# Predictor0916 — Google Colab GPU 训练

本 Notebook 复用 `scripts/test_training.py` 的完整训练入口，并使用 Colab 的 CUDA GPU。运行前请在 Colab 菜单中选择 **运行时 → 更改运行时类型 → T4 GPU（或其他 GPU）**。

仓库需要位于 `/content/Predictor0916`，或者位于 `/content` 下某个目录的第一层，例如 `/content/paper0910/Predictor0916`。如果仓库尚未上传，请先将整个 `Predictor0916` 目录上传到 Colab；如果仓库有可访问的 Git URL，也可以先运行 `!git clone <repository-url>`。

## 1. 安装依赖

Colab 已预装 CUDA 版 PyTorch，因此这里不重新安装 PyTorch。

In [ ]:
%pip install -q transformers datasets pandas numpy pyarrow accelerate sentencepiece huggingface_hub

## 2. 检查 GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 CUDA GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → GPU。"
    )

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 3. 定位项目并导入训练入口

如果自动搜索没有找到项目，请直接修改 `PROJECT_DIR`。

In [ ]:
from pathlib import Path
import sys

project_candidates = [
    Path("/content/Predictor0916"),
    Path.cwd() / "Predictor0916",
    *Path("/content").glob("*/Predictor0916"),
]
PROJECT_DIR = next((path.resolve() for path in project_candidates if path.is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "找不到 Predictor0916。请上传/克隆项目，或手动设置 PROJECT_DIR。"
    )

REPO_ROOT = PROJECT_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Predictor0916.scripts.test_training import main as training_main

print("Project:", PROJECT_DIR)

## 4. Hugging Face 登录

默认 Llama 模型可能需要授权。推荐在 Colab 左侧 **Secrets** 中创建名为 `HF_TOKEN` 的 secret 并允许 Notebook 访问。下面的代码不会打印 token。若模型已位于本地且不需要认证，可以跳过登录。

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("HF_TOKEN 未配置；仅公开模型或已有本地模型可直接加载。")

## 5. 配置实验

如果 `DATA_PATH` 不存在，训练入口会自动下载 ForeLen，并使用指定 LLM 生成隐藏状态和回答长度。完整预处理可能耗时很长。Colab T4 建议使用 `float16`；仅在确认 GPU 支持 BF16 时改为 `bfloat16`。

In [ ]:
# LLM configuration
MODEL_ID_OR_PATH = "meta-llama/Llama-3.2-1B-Instruct"
LLM_DEVICE = "cuda:0"
MLP_DEVICE = "cuda:0"
TORCH_DTYPE = "float16"
LLM_BATCH_SIZE = 1
MAX_PROMPT_LENGTH = None
TRUST_REMOTE_CODE = True

# ForeLen config must be chosen explicitly when changing the LLM. For
# Qwen2.5-0.5B RL data, use: DATASET_SUBSET = "qwen2.5-0.5b-rl"
DATASET_SUBSET = "llama3.2-1b-rl"

# Data and output paths. Point DATA_PATH to an existing processed Parquet
# to skip LLM feature generation. Colab /content storage is temporary.
DATA_PATH = PROJECT_DIR / "data" / "llama3.2-1b-rl-generated.parquet"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "test_training_colab"
LOAD_CHECKPOINT = None

# MLP and training configuration
VALIDATION_RATIO = 0.2
NUM_BINS = 20
TARGET_QUANTILES = (0.01, 0.99)
LOSS_TYPE = "soft_label"
LAMBDA_VAL = 0.95
EPOCHS = 10
TRAIN_BATCH_SIZE = 256
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
PATIENCE = 3  # Set to 0 to disable early stopping.
SEED = 42
NUM_WORKERS = 2
PIN_MEMORY = True

### 可选：将数据和输出放在 Google Drive

Colab 的 `/content` 会在运行时结束后清空。如需持久保存，先挂载 Drive，再将上面的 `DATA_PATH` 和 `OUTPUT_DIR` 改到 `/content/drive/MyDrive/...`。

In [ ]:
# Uncomment when Google Drive persistence is needed.
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_PATH = Path("/content/drive/MyDrive/Predictor0916/data/llama3.2-1b-rl-generated.parquet")
# OUTPUT_DIR = Path("/content/drive/MyDrive/Predictor0916/outputs/test_training_colab")

## 6. 运行完整训练流水线

In [ ]:
training_args = [
    "--model-id-or-path", str(MODEL_ID_OR_PATH),
    "--llm-device", LLM_DEVICE,
    "--device", MLP_DEVICE,
    "--torch-dtype", TORCH_DTYPE,
    "--llm-batch-size", str(LLM_BATCH_SIZE),
    "--dataset-subset", DATASET_SUBSET,
    "--data-path", str(DATA_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--validation-ratio", str(VALIDATION_RATIO),
    "--num-bins", str(NUM_BINS),
    "--target-quantiles", str(TARGET_QUANTILES[0]), str(TARGET_QUANTILES[1]),
    "--loss-type", LOSS_TYPE,
    "--lambda-val", str(LAMBDA_VAL),
    "--epochs", str(EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--patience", str(PATIENCE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
]
if MAX_PROMPT_LENGTH is not None:
    training_args.extend(["--max-prompt-length", str(MAX_PROMPT_LENGTH)])
if TRUST_REMOTE_CODE:
    training_args.append("--trust-remote-code")
else:
    training_args.append("--no-trust-remote-code")
if PIN_MEMORY:
    training_args.append("--pin-memory")
if LOAD_CHECKPOINT is not None:
    training_args.extend(["--load-checkpoint", str(LOAD_CHECKPOINT)])

record = training_main(training_args)

## 7. 查看结果

In [ ]:
import json
import pandas as pd
from IPython.display import display

print("Split sizes:")
print(json.dumps(record["split_sizes"], indent=2, ensure_ascii=False))
print("Validation metrics:")
print(json.dumps(record["metrics"], indent=2, ensure_ascii=False))
print("Artifacts:")
print(json.dumps(record["artifacts"], indent=2, ensure_ascii=False))

predictions = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")
display(predictions.head(10))

## 8. 可选：打包并下载输出

如果输出已经保存在 Google Drive，则不需要执行此步骤。

In [ ]:
# Uncomment to download a ZIP archive from Colab.
# import shutil
# from google.colab import files
# archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
# files.download(archive_path)